In [1]:
# Load environment and suppress warnings

import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-5-mini", temperature=0)
else:
    model = None
    print("Skipping model init. No API key found.")


In [3]:
# Two simple target chains

billing_prompt = PromptTemplate.from_template(
"""
You are a billing support assistant.

Write one sentence telling the customer what billing info you need next.

Ticket:
{ticket_text}
"""
)

technical_prompt = PromptTemplate.from_template(
"""
You are a technical support assistant.

Write one sentence telling the customer what technical detail you need next.

Ticket:
{ticket_text}
"""
)

billing_chain = billing_prompt | model | StrOutputParser()
technical_chain = technical_prompt | model | StrOutputParser()

In [4]:
# Router decides which chain should handle the ticket

router_prompt = PromptTemplate.from_template(
"""
Classify the support ticket into exactly one route:

Billing
Technical

Return only the route name.

Ticket:
{ticket_text}
"""
)

router_chain = router_prompt | model | StrOutputParser()

In [5]:
# Route a ticket to the correct chain

ticket_text = "Customer reports being charged twice for the same subscription this month."

route = router_chain.invoke({"ticket_text": ticket_text}).strip()

print("=== ROUTE ===")
print(route)
print()

if route == "Billing":
    result = billing_chain.invoke({"ticket_text": ticket_text})
else:
    result = technical_chain.invoke({"ticket_text": ticket_text})

print("=== RESULT ===")
print(result)

=== ROUTE ===
Billing

=== RESULT ===
Please reply with your account email or customer ID, the subscription or invoice ID (or name), and a screenshot or copy of your bank statement showing the two charges (including dates, amounts, and the last four digits of the card) so we can investigate.


In [6]:
# Route a different ticket

ticket_text_2 = "User cannot login and gets an error after resetting the password."

route_2 = router_chain.invoke({"ticket_text": ticket_text_2}).strip()

print("=== ROUTE 2 ===")
print(route_2)
print()

if route_2 == "Billing":
    result_2 = billing_chain.invoke({"ticket_text": ticket_text_2})
else:
    result_2 = technical_chain.invoke({"ticket_text": ticket_text_2})

print("=== RESULT 2 ===")
print(result_2)

=== ROUTE 2 ===
Technical

=== RESULT 2 ===
Please send the exact error message (or a screenshot), the account email/username, and the device/OS plus whether you're using the web or mobile app and its browser/app version.
